In [1]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

model_name = "BTCUSD-5m"
src_path = folder_path / "clean" / f"{model_name}-v6-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR,hl_pct
0,1704066000000,42241.089844,60.923592,67.243118,1704066000000,0,0,0.000237,0.696150,0.000237,0.001592,1704065400000,1704064500000,0,0,23,-0.011206,-0.011908,-0.011206,0.000710
1,1704066300000,42269.828125,84.640320,66.823570,1704066300000,1,1,0.000680,0.963925,0.000680,0.001581,1704066300000,1704064500000,0,0,24,-0.010444,-0.011446,-0.010533,0.001013
2,1704066600000,42222.101562,76.113548,67.325706,1704066600000,1,1,-0.001129,0.863871,-0.001129,0.001595,1704066300000,1704064500000,0,0,25,-0.010533,-0.011671,-0.011650,0.001151
3,1704066900000,42283.578125,81.403389,66.089752,1704066900000,1,1,0.001456,0.919758,0.001456,0.001563,1704066300000,1704064500000,0,0,26,-0.010211,-0.011650,-0.010211,0.001454
4,1704067200000,42397.230469,155.257309,67.278091,1704067200000,1,1,0.002688,1.737683,0.002688,0.001587,1704067200000,1704067200000,-1,0,27,-0.007551,-0.010739,-0.007551,0.003213


In [2]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(228677, 20)
Index(['timestamp', 'close', 'vol', 'atr42', 'ts_5m', 'label', 'train_mask',
       'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'ts_15m', 'ts_45m',
       '15STR_confirmed', '45STR_confirmed', 'barsSince45STR', 'hDTK_45STR',
       'lDTK_45STR', 'cDTK_45STR', 'hl_pct'],
      dtype='object')


,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR,hl_pct
0,1704066000000,42241.089844,60.923592,67.243118,1704066000000,0,0,0.000237,0.696150,0.000237,0.001592,1704065400000,1704064500000,0,0,23,-0.011206,-0.011908,-0.011206,0.000710
1,1704066300000,42269.828125,84.640320,66.823570,1704066300000,1,1,0.000680,0.963925,0.000680,0.001581,1704066300000,1704064500000,0,0,24,-0.010444,-0.011446,-0.010533,0.001013
2,1704066600000,42222.101562,76.113548,67.325706,1704066600000,1,1,-0.001129,0.863871,-0.001129,0.001595,1704066300000,1704064500000,0,0,25,-0.010533,-0.011671,-0.011650,0.001151
3,1704066900000,42283.578125,81.403389,66.089752,1704066900000,1,1,0.001456,0.919758,0.001456,0.001563,1704066300000,1704064500000,0,0,26,-0.010211,-0.011650,-0.010211,0.001454
4,1704067200000,42397.230469,155.257309,67.278091,1704067200000,1,1,0.002688,1.737683,0.002688,0.001587,1704067200000,1704067200000,-1,0,27,-0.007551,-0.010739,-0.007551,0.003213


In [3]:
# tactical drop extra timeframe and non-stationary data
# prepare for scaler : scaler is order strict
df.drop(columns=["close", "vol", "atr42","ts_5m", "ts_15m", "ts_45m"], inplace=True)
print(df.shape)
df.head()

(228677, 14)


,timestamp,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR,hl_pct
0,1704066000000,0,0,0.000237,0.696150,0.000237,0.001592,0,0,23,-0.011206,-0.011908,-0.011206,0.000710
1,1704066300000,1,1,0.000680,0.963925,0.000680,0.001581,0,0,24,-0.010444,-0.011446,-0.010533,0.001013
2,1704066600000,1,1,-0.001129,0.863871,-0.001129,0.001595,0,0,25,-0.010533,-0.011671,-0.011650,0.001151
3,1704066900000,1,1,0.001456,0.919758,0.001456,0.001563,0,0,26,-0.010211,-0.011650,-0.010211,0.001454
4,1704067200000,1,1,0.002688,1.737683,0.002688,0.001587,-1,0,27,-0.007551,-0.010739,-0.007551,0.003213


# At this point

- scale all cols - pick training case later

In [4]:
# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)
n = len(df)

# Chronological split on ALL rows (includes timeout rows — preserves time boundaries)
train_all = df.iloc[:int(n*0.8)]
val_all   = df.iloc[int(n*0.8):int(n*0.9)]
test_all  = df.iloc[int(n*0.9):]

# Apply train_mask: filter to trainable rows (label 1 or -1) within each split
train_df = train_all[train_all['train_mask'] == 1].copy()
val_df   = val_all[val_all['train_mask'] == 1].copy()
test_df  = test_all[test_all['train_mask'] == 1].copy()

print(f"All rows  — Train: {len(train_all):,} | Val: {len(val_all):,} | Test: {len(test_all):,}")
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


All rows  — Train: 182,941 | Val: 22,868 | Test: 22,868
Trainable — Train: 123,609 | Val: 15,340 | Test: 15,044


# However — there's a catch for your case. You're using a chronological split

In [5]:
print(df.columns)
print(len(df.columns))
del df

Index(['timestamp', 'label', 'train_mask', 'body_pct', 'vol_norm',
       'close_ret1', 'atr42_pct', '15STR_confirmed', '45STR_confirmed',
       'barsSince45STR', 'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR', 'hl_pct'],
      dtype='object')
14


In [6]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

# scale all feature columns — pick subset at training time
scale_cols = [
    'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct',
    'barsSince45STR',
    'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR',
    'hl_pct',
]

# not scaled: timestamp, ts_45m, label, train_mask,
#             15STR_confirmed, 45STR_confirmed  (categorical: -1/0/1)


scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
val_df[scale_cols]   = scaler.transform(val_df[scale_cols])        # transform only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler-V6.pkl")
print("Scaler saved.")
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")


Scaler saved.
Train: (123609, 14) | Val: (15340, 14) | Test: (15044, 14)


In [7]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train-v6.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val-v6.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test-v6.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSD-5m-train-v6.jsonl
Saved val: 202603-BTCUSD-5m-val-v6.jsonl
Saved test:  202603-BTCUSD-5m-test-v6.jsonl


In [9]:
non_feature = ['timestamp', 'label', 'train_mask']
feature_cols = [c for c in train_df.columns if c not in non_feature]

print(f"Train columns  : {len(train_df.columns)}  → {list(train_df.columns)}")
print(f"Feature columns: {len(feature_cols)}  → {feature_cols}")
print(f"Scaled columns : {len(scale_cols)}  → {scale_cols}")


Train columns  : 14  → ['timestamp', 'label', 'train_mask', 'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', '15STR_confirmed', '45STR_confirmed', 'barsSince45STR', 'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR', 'hl_pct']
Feature columns: 11  → ['body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', '15STR_confirmed', '45STR_confirmed', 'barsSince45STR', 'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR', 'hl_pct']
Scaled columns : 9  → ['body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'barsSince45STR', 'hDTK_45STR', 'lDTK_45STR', 'cDTK_45STR', 'hl_pct']
